In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import math
import numpy as np
import pickle
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score
import scipy.signal as signal
from sklearn.metrics import roc_curve, auc


In [2]:
df0_0 = pd.read_csv('./bdhsc_2024/stage1_labeled/0_0.csv')
df0_1 = pd.read_csv('./bdhsc_2024/stage1_labeled/0_1.csv')
df1_0 = pd.read_csv('./bdhsc_2024/stage1_labeled/1_0.csv')
df1_1 = pd.read_csv('./bdhsc_2024/stage1_labeled/1_1.csv')
df2_0 = pd.read_csv('./bdhsc_2024/stage1_labeled/2_0.csv')
df2_1 = pd.read_csv('./bdhsc_2024/stage1_labeled/2_1.csv')
df3_0 = pd.read_csv('./bdhsc_2024/stage1_labeled/3_0.csv')
df3_1 = pd.read_csv('./bdhsc_2024/stage1_labeled/3_1.csv')
df4_0 = pd.read_csv('./bdhsc_2024/stage1_labeled/4_0.csv')
df4_1 = pd.read_csv('./bdhsc_2024/stage1_labeled/4_1.csv')
df5_0 = pd.read_csv('./bdhsc_2024/stage1_labeled/5_0.csv')
df5_1 = pd.read_csv('./bdhsc_2024/stage1_labeled/5_1.csv')
df6_0 = pd.read_csv('./bdhsc_2024/stage1_labeled/6_0.csv')
df6_1 = pd.read_csv('./bdhsc_2024/stage1_labeled/6_1.csv')
df7_0 = pd.read_csv('./bdhsc_2024/stage1_labeled/7_0.csv')
df7_1 = pd.read_csv('./bdhsc_2024/stage1_labeled/7_1.csv')



In [3]:
df0_0['recording_id'] = 0
df0_1['recording_id'] = 1
df1_0['recording_id'] = 0
df1_1['recording_id'] = 1
df2_0['recording_id'] = 0
df2_1['recording_id'] = 1
df3_0['recording_id'] = 0
df3_1['recording_id'] = 1
df4_0['recording_id'] = 0
df4_1['recording_id'] = 1
df5_0['recording_id'] = 0
df5_1['recording_id'] = 1
df6_0['recording_id'] = 0
df6_1['recording_id'] = 1
df7_0['recording_id'] = 0
df7_1['recording_id'] = 1

In [4]:
df0_0['index'] = df0_0.index
df0_1['index'] = df0_1.index
df1_0['index']= df1_0.index
df1_1['index']= df1_1.index
df2_0['index']= df2_0.index
df2_1['index']= df2_1.index
df3_0['index']= df3_0.index
df3_1['index']= df3_1.index
df4_0['index']= df4_0.index
df4_1['index']= df4_1.index
df5_0['index']= df5_0.index
df5_1['index']= df5_1.index
df6_0['index']= df6_0.index
df6_1['index']= df6_1.index
df7_0['index']= df7_0.index
df7_1['index']= df7_1.index

In [5]:
df1 = pd.concat([df0_0, df0_1])
df2 = pd.concat([df1_0, df1_1])
df3 = pd.concat([df2_0, df2_1])
df4 = pd.concat([df3_0, df3_1])
df5 = pd.concat([df4_0, df4_1])
df6 = pd.concat([df5_0, df5_1])
df7 = pd.concat([df6_0, df6_1])
df8 = pd.concat([df7_0, df7_1])


### Data Preprocessing

In [6]:
## Rename the labels
df1['label'] = df1['5000']
df1 = df1.drop('5000', axis=1)

df2['label'] = df2['5000']
df2 = df2.drop('5000', axis=1)

df3['label'] = df3['5000']
df3 = df3.drop('5000', axis=1)

df4['label'] = df4['5000']
df4 = df4.drop('5000', axis=1)

df5['label'] = df5['5000']
df5 = df5.drop('5000', axis=1)

df6['label'] = df6['5000']
df6 = df6.drop('5000', axis=1)

df7['label'] = df7['5000']
df7 = df7.drop('5000', axis=1)

df8['label'] = df8['5000']
df8 = df8.drop('5000', axis=1)


In [7]:
## Add identifer for each animal
df1['animal'] = 0
df2['animal'] = 1
df3['animal'] = 2
df4['animal'] = 3
df5['animal'] = 4
df6['animal'] = 5
df7['animal'] = 6
df8['animal'] = 7

In [8]:
df_temp = pd.concat([df1, df2, df3, df4, df5, df6, df7, df8 ])

In [10]:
df_temp.head()

,0,1,2,3,4,5,6,7,8,9,...,4994,4995,4996,4997,4998,4999,recording_id,index,label,animal
0,-9.079118e-06,5.874723e-06,1.853971e-05,2.021820e-05,9.842069e-06,-5.722133e-06,-1.426719e-05,-9.231709e-06,-6.866560e-07,-2.822919e-06,...,-9.132524e-05,-9.224079e-05,-6.752117e-05,-3.349355e-05,-1.136797e-05,6.637675e-06,0,0,2,0
1,3.150988e-05,3.944457e-05,6.485084e-06,-3.837644e-05,-4.768444e-05,-2.601663e-05,-1.441978e-05,-2.113375e-05,-1.945525e-05,-4.501411e-06,...,-4.463264e-05,-2.388037e-05,-1.396200e-05,-2.555886e-05,-3.776608e-05,-2.677958e-05,0,1,2,0
2,1.144427e-06,1.976043e-05,1.426719e-05,-5.874723e-06,-2.296483e-05,-2.525368e-05,-9.536889e-06,1.380941e-05,2.571145e-05,1.441978e-05,...,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,0,2,2,0
3,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,...,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,0,3,2,0
4,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,...,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,0,4,2,0


In [11]:
del df1, df2, df3, df4, df5, df6, df7, df8
del df0_0,df0_1,df1_0,df1_1,df2_0,df2_1,df3_0,df3_1,df4_0,df4_1,df5_0,df5_1,df6_0,df6_1,df7_0,df7_1,

### Feature Engineering

In [12]:
def calculate_signal_strength(row):
# Simulated EEG data (replace this with your actual EEG data)
  sampling_rate = 500  # Hz
  duration = 10  # seconds
  num_samples = sampling_rate * duration
  time = np.linspace(0, duration, num_samples)
  eeg_data = row[:num_samples] # Replace with your actual EEG data

  # Apply FFT to get the frequency spectrum
  fft_values = np.fft.fft(eeg_data)
  freqs = np.fft.fftfreq(len(fft_values), 1/sampling_rate)
  power_spectrum = np.abs(fft_values) ** 2

  # Define frequency bands
  delta_band = (0.5, 4)
  theta_band = (4, 8)
  alpha_band = (8, 13)
  beta_band = (13, 30)
  gamma_band = (30, 100)

  # Calculate power within each frequency band
  delta_power = np.sum(power_spectrum[(freqs >= delta_band[0]) & (freqs <= delta_band[1])])
  theta_power = np.sum(power_spectrum[(freqs >= theta_band[0]) & (freqs <= theta_band[1])])
  alpha_power = np.sum(power_spectrum[(freqs >= alpha_band[0]) & (freqs <= alpha_band[1])])
  beta_power = np.sum(power_spectrum[(freqs >= beta_band[0]) & (freqs <= beta_band[1])])
  gamma_power = np.sum(power_spectrum[(freqs >= gamma_band[0]) & (freqs <= gamma_band[1])])

  row['delta_power'] = delta_power
  row['theta_power'] = theta_power
  row['alpha_power'] = alpha_power
  row['beta_power'] = beta_power
  row['gamma_power'] = gamma_power

  return delta_power, theta_power, alpha_power, beta_power, gamma_power



In [13]:
df_temp[['delta_power', 'theta_power', 'alpha_power', 'beta_power', 'gamma_power']] = df_temp.apply(calculate_signal_strength, axis=1, result_type='expand')

In [14]:
def calculate_mmd_np(data):
  """
  Calculates the Maximum Minimum Distance (MMD) for a given EEG data segment.

  Args:
    data: A NumPy array of shape (5000,) representing the EEG data segment.

  Returns:
    A float representing the MMD value for the segment.
  """

  mmd = 0
  data = np.array(data)
  segments = data.reshape(-1, 500)  # Reshape data into 10 segments of 500 points each

  for segment in segments:
    min_idx = np.argmin(segment)
    max_idx = np.argmax(segment)
    min_value = segment[min_idx]
    max_value = segment[max_idx]
    mmd += np.sqrt(((max_idx - min_idx) ** 2) + ((max_value - min_value) ** 2))

  return mmd

In [15]:
df_temp['mmd'] = df_temp.iloc[:, :5000].apply(calculate_mmd_np, axis=1, )

In [16]:
df_temp.head()

,0,1,2,3,4,5,6,7,8,9,...,recording_id,index,label,animal,delta_power,theta_power,alpha_power,beta_power,gamma_power,mmd
0,-9.079118e-06,5.874723e-06,1.853971e-05,2.021820e-05,9.842069e-06,-5.722133e-06,-1.426719e-05,-9.231709e-06,-6.866560e-07,-2.822919e-06,...,0,0,2,0,4.398571e-03,4.171663e-03,1.944315e-03,1.633690e-03,3.856505e-03,2790.0
1,3.150988e-05,3.944457e-05,6.485084e-06,-3.837644e-05,-4.768444e-05,-2.601663e-05,-1.441978e-05,-2.113375e-05,-1.945525e-05,-4.501411e-06,...,0,1,2,0,5.031234e-03,6.190507e-03,2.447044e-03,1.628200e-03,3.875399e-03,1494.0
2,1.144427e-06,1.976043e-05,1.426719e-05,-5.874723e-06,-2.296483e-05,-2.525368e-05,-9.536889e-06,1.380941e-05,2.571145e-05,1.441978e-05,...,0,2,2,0,4.053951e-04,6.670872e-04,7.339369e-04,1.683663e-04,4.988374e-04,422.0
3,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,...,0,3,2,0,1.449288e-40,3.053411e-41,2.838300e-41,7.685703e-42,5.727078e-42,0.0
4,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,-7.629511e-08,...,0,4,2,0,1.449288e-40,3.053411e-41,2.838300e-41,7.685703e-42,5.727078e-42,0.0


In [19]:
x = df_temp.drop(['label', 'index', 'recording_id'], axis=1)
y = df_temp['label']

In [20]:
x.shape, y.shape

((138240, 5009), (138240,))

#### Baseline Model

In [22]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

model = LogisticRegression()
# Train the model on the training data
model.fit(X_train, y_train)

/opt/homebrew/Caskroom/miniforge/base/envs/.venv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:763: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [23]:
# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Calculate precision
precision = precision_score(y_test, y_pred, average='macro')  # Macro-averaging

# Calculate recall
recall = recall_score(y_test, y_pred, average='macro')  # Macro-averaging

# Calculate F1-score
f1 = f1_score(y_test, y_pred, average='macro')  # Macro-averaging

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

Accuracy: 0.5491898148148148
Precision: 0.36136651525519753
Recall: 0.37882148058787507
F1-score: 0.3658044424595241


/opt/homebrew/Caskroom/miniforge/base/envs/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


### XGBoost (without hyperparameter tuning)

In [27]:
# Split the data into train and test sets (using 20% for testing)
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)

# Create the XGBoost classifier model
model = xgb.XGBClassifier(
  num_class=3, 
)

# Train the model
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)


In [28]:
# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Calculate precision
precision = precision_score(y_test, y_pred, average='macro')  # Macro-averaging

# Calculate recall
recall = recall_score(y_test, y_pred, average='macro')  # Macro-averaging

# Calculate F1-score
f1 = f1_score(y_test, y_pred, average='macro')  # Macro-averaging

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

Accuracy: 0.9152199074074074
Precision: 0.8650570243137584
Recall: 0.8136630542683086
F1-score: 0.8352304234755638


### XGBoost (Parameters Tuning)

In [36]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, stratify=y, random_state=42)

In [38]:
# Create an XGBoost classifier
xgb_model = xgb.XGBClassifier(
    objective='multi:softmax',  # Multiclass classification
    num_class=3,  # Number of classes in the target variable
    learning_rate=0.1,
    n_estimators=500,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0,
    reg_lambda=1,
    random_state=42,
)

xgb_model.fit(X_train, y_train)


In [ ]:
# Make predictions on the test set
y_pred = xgb_model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Calculate precision
precision = precision_score(y_test, y_pred, average='macro')  # Macro-averaging

# Calculate recall
recall = recall_score(y_test, y_pred, average='macro')  # Macro-averaging

# Calculate F1-score
f1 = f1_score(y_test, y_pred, average='macro')  # Macro-averaging

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

Accuracy: 0.9188368055555556
Precision: 0.8728965576020439
Recall: 0.8260361768161685
F1-score: 0.8461915029853783


In [56]:
# Make predictions on the Train set
y_pred = model.predict(X_train)

# Calculate accuracy
accuracy = accuracy_score(y_train, y_pred)
print("Accuracy:", accuracy)

# Calculate precision
precision = precision_score(y_train, y_pred, average='macro')  # Macro-averaging

# Calculate recall
recall = recall_score(y_train, y_pred, average='macro')  # Macro-averaging

# Calculate F1-score
f1 = f1_score(y_train, y_pred, average='macro')  # Macro-averaging

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

Accuracy: 0.9391741071428571
Precision: 0.9198711822692194
Recall: 0.8659845250088071
F1-score: 0.889305297613165
